# CENG 467 — Question 1: Representation Learning in Text Classification

**Author:** Furkan  
**Course:** CENG 467 — Natural Language Understanding and Generation  
**Instructor:** Prof. Dr. Aytuğ Onan

## Objective
Compare three representation learning paradigms — *sparse* (TF-IDF), *dense* (BiLSTM with learned embeddings), and *contextual* (DistilBERT) — on the SST-2 binary sentiment classification task.

## Setup
- **Runtime:** Google Colab — GPU (T4)
- **Random Seed:** 42
- **Dataset:** SST-2 (Stanford Sentiment Treebank, binary)
- **Metrics:** Accuracy, Macro-F1

Run `Runtime → Run all` after selecting GPU.

## 1. Environment Setup

In [ ]:
!pip install -q datasets==2.19.0 transformers==4.41.0 scikit-learn==1.4.2 nltk==3.8.1 accelerate==0.30.0

In [ ]:
import os, random, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
warnings.filterwarnings('ignore')

# === Reproducibility ===
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

import nltk
nltk.download('punkt', quiet=True); nltk.download('stopwords', quiet=True)

os.makedirs('results', exist_ok=True)

## 2. Dataset: SST-2

SST-2 (Stanford Sentiment Treebank) is a binary sentiment classification benchmark from the GLUE suite. Each example is a movie-review sentence labeled as positive (1) or negative (0). Splits: 67,349 train / 872 validation / 1,821 test (test labels are private; we use the validation set as our held-out test set, following standard GLUE evaluation practice).

In [ ]:
from datasets import load_dataset
ds = load_dataset('glue', 'sst2')

# We split train into train+dev (90/10), keep 'validation' as our test set.
train_full = ds['train'].shuffle(seed=SEED)
split = train_full.train_test_split(test_size=0.1, seed=SEED)
train_ds, dev_ds, test_ds = split['train'], split['test'], ds['validation']

print(f'Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}')
print('Example:', train_ds[0])
print('Label distribution (train):', np.bincount(train_ds['label']))

## 3. Preprocessing & Tokenization Comparison

We compare two tokenization strategies for the classical model:
1. **Whitespace + lowercase** (minimal)
2. **NLTK word_tokenize + lowercase + stopword removal**

The neural and transformer models use their own tokenizers (Keras-style word-index for BiLSTM, WordPiece for DistilBERT).

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

STOPWORDS = set(stopwords.words('english'))

def tokenize_simple(text):
    return text.lower().split()

def tokenize_nltk(text, remove_stop=True):
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    toks = word_tokenize(text)
    if remove_stop:
        toks = [t for t in toks if t not in STOPWORDS and len(t) > 1]
    return toks

sample = train_ds[0]['sentence']
print('Original :', sample)
print('Simple   :', tokenize_simple(sample))
print('NLTK     :', tokenize_nltk(sample))

## 4. Model 1 — TF-IDF + Logistic Regression (Sparse Representation)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

X_train_txt = [x['sentence'] for x in train_ds]
y_train     = np.array(train_ds['label'])
X_test_txt  = [x['sentence'] for x in test_ds]
y_test      = np.array(test_ds['label'])

results_tfidf = {}
for name, tokenizer in [('simple', tokenize_simple), ('nltk_nostop', tokenize_nltk)]:
    t0 = time.time()
    vec = TfidfVectorizer(tokenizer=tokenizer, ngram_range=(1,2), max_features=50000, min_df=2)
    Xtr = vec.fit_transform(X_train_txt)
    Xte = vec.transform(X_test_txt)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
    clf.fit(Xtr, y_train)
    y_pred = clf.predict(Xte)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    results_tfidf[name] = {'accuracy': acc, 'macro_f1': f1, 'time_sec': time.time()-t0}
    print(f'[{name:12s}] Acc={acc:.4f}  Macro-F1={f1:.4f}  ({time.time()-t0:.1f}s)')

best = max(results_tfidf, key=lambda k: results_tfidf[k]['macro_f1'])
print(f'\nBest TF-IDF tokenizer: {best}')
tfidf_final = results_tfidf[best]
tfidf_final['best_tokenizer'] = best

# Save preds for error analysis
vec = TfidfVectorizer(tokenizer=tokenize_simple if best=='simple' else tokenize_nltk, ngram_range=(1,2), max_features=50000, min_df=2)
Xtr = vec.fit_transform(X_train_txt); Xte = vec.transform(X_test_txt)
clf_final = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xtr, y_train)
tfidf_preds = clf_final.predict(Xte)

## 5. Model 2 — BiLSTM (Dense, Learned Embeddings)

In [ ]:
from collections import Counter

PAD, UNK = 0, 1
MAX_LEN = 50

# Build vocab from training set
tokenized_train = [tokenize_simple(t) for t in X_train_txt]
counter = Counter(t for sent in tokenized_train for t in sent)
vocab = {'<pad>': PAD, '<unk>': UNK}
for w, c in counter.most_common(20000):
    if c >= 2:
        vocab[w] = len(vocab)
VOCAB_SIZE = len(vocab)
print(f'Vocab size: {VOCAB_SIZE}')

def encode(text):
    ids = [vocab.get(t, UNK) for t in tokenize_simple(text)][:MAX_LEN]
    ids += [PAD] * (MAX_LEN - len(ids))
    return ids

class TextDS(Dataset):
    def __init__(self, texts, labels):
        self.x = torch.tensor([encode(t) for t in texts], dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

train_loader = DataLoader(TextDS(X_train_txt, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(TextDS(X_test_txt,  y_test),  batch_size=64)

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128, num_classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True, num_layers=1)
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden*2, num_classes)
    def forward(self, x):
        e = self.emb(x)
        out, _ = self.lstm(e)
        h = out.mean(dim=1)               # mean-pool
        return self.fc(self.drop(h))

model = BiLSTMClassifier(VOCAB_SIZE).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 5
t0 = time.time()
for epoch in range(EPOCHS):
    model.train(); total = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward(); opt.step()
        total += loss.item() * x.size(0)
    print(f'Epoch {epoch+1}: train_loss={total/len(train_loader.dataset):.4f}')

model.eval(); preds = []
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(DEVICE)).argmax(dim=1).cpu().numpy()
        preds.extend(p)
lstm_preds = np.array(preds)
lstm_acc = accuracy_score(y_test, lstm_preds)
lstm_f1  = f1_score(y_test, lstm_preds, average='macro')
lstm_time = time.time() - t0
print(f'\nBiLSTM | Acc={lstm_acc:.4f}  Macro-F1={lstm_f1:.4f}  ({lstm_time:.1f}s)')

## 6. Model 3 — DistilBERT (Contextual Representation)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset

MODEL_NAME = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def hf_encode(batch):
    return tok(batch['sentence'], padding='max_length', truncation=True, max_length=64)

hf_train = HFDataset.from_dict({'sentence': X_train_txt, 'label': y_train.tolist()}).map(hf_encode, batched=True)
hf_test  = HFDataset.from_dict({'sentence': X_test_txt,  'label': y_test.tolist() }).map(hf_encode, batched=True)

hf_train.set_format('torch', columns=['input_ids','attention_mask','label'])
hf_test.set_format ('torch', columns=['input_ids','attention_mask','label'])

model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

args = TrainingArguments(
    output_dir='distilbert_out',
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=200,
    save_strategy='no',
    seed=SEED,
    report_to='none',
)

def compute_metrics(p):
    preds = p.predictions.argmax(axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds),
            'macro_f1': f1_score(p.label_ids, preds, average='macro')}

trainer = Trainer(model=model_bert, args=args, train_dataset=hf_train,
                  eval_dataset=hf_test, compute_metrics=compute_metrics)

t0 = time.time()
trainer.train()
bert_eval = trainer.evaluate()
bert_time = time.time() - t0
bert_preds = trainer.predict(hf_test).predictions.argmax(axis=1)
print(bert_eval)

## 7. Final Results & Error Analysis

In [ ]:
summary = pd.DataFrame([
    {'Model': 'TF-IDF + LogReg',  'Accuracy': tfidf_final['accuracy'], 'Macro-F1': tfidf_final['macro_f1'], 'Train Time (s)': tfidf_final['time_sec']},
    {'Model': 'BiLSTM',           'Accuracy': lstm_acc,                'Macro-F1': lstm_f1,                'Train Time (s)': lstm_time},
    {'Model': 'DistilBERT',       'Accuracy': bert_eval['eval_accuracy'], 'Macro-F1': bert_eval['eval_macro_f1'], 'Train Time (s)': bert_time},
])
print(summary.to_string(index=False))
summary.to_csv('results/q1_summary.csv', index=False)

In [ ]:
# Error analysis: examples DistilBERT got wrong
wrong_idx = np.where(bert_preds != y_test)[0]
print(f'DistilBERT misclassified {len(wrong_idx)} / {len(y_test)} examples ({len(wrong_idx)/len(y_test)*100:.1f}%)')
print('\n--- 5 sample errors ---')
errors = []
for i in wrong_idx[:5]:
    err = {'sentence': X_test_txt[i], 'true': int(y_test[i]), 'pred_bert': int(bert_preds[i]),
           'pred_lstm': int(lstm_preds[i]), 'pred_tfidf': int(tfidf_preds[i])}
    errors.append(err)
    print(f"\n[true={err['true']} | bert={err['pred_bert']} | lstm={err['pred_lstm']} | tfidf={err['pred_tfidf']}]")
    print(f"  {err['sentence']}")

with open('results/q1_errors.json','w') as f:
    json.dump(errors, f, indent=2)
with open('results/q1_full.json','w') as f:
    json.dump({'tfidf': tfidf_final, 'bilstm': {'accuracy': lstm_acc, 'macro_f1': lstm_f1, 'time_sec': lstm_time},
               'distilbert': {'accuracy': bert_eval['eval_accuracy'], 'macro_f1': bert_eval['eval_macro_f1'], 'time_sec': bert_time}}, f, indent=2)
print('\nResults saved to results/')

## 8. Discussion (in Report)

See `reports/CENG467_Midterm_Report.pdf` for the full analysis comparing sparse, dense, and contextual representations in terms of:
- Performance (accuracy, macro-F1)
- Training cost
- Interpretability
- Common error patterns (negation, sarcasm, comparative phrasing)